In [ ]:
# -*- coding: utf-8 -*-
"""
Clone-only extractor for Android instrumentation-testing related files.

What this does now:
- Auto-detect default branch via: git ls-remote --symref <repo> HEAD
- Shallow clone that branch (--depth 1)
- Extract BOTH config files AND test sources
- YAML files: **only** those matching strict CI locations in `ci_patterns` are saved.
- flat filenames: {owner}.{repo}__{ci_platform}++{file_lower}
- Split into two buckets:
    - All_Config_Files: Gradle & settings (any name) scripts, gradle.properties, **version catalogs**,
      **convention plugins** (buildSrc/**, gradle/plugins/**) including .kt/.java plugin code,
      AndroidManifest.xml (any source set), CI YAMLs, SH, CI-ish JSON (under workflows or known names)
    - All_Test_Files: **ONLY** files under src/androidTest/** (kt/java/xml/etc.)
- CSV index (no ci_source now): owner, repo, repo_url, default_branch, commit_sha,
  relative_path, filename, flat_filename, ci_platform, html_url, saved_to, bucket, components
- Project metadata via GitHub API + paginated counts (contributors, pulls, commits)
- Per-commit CSV with "touches_androidTest / touches_github_workflows / touches_gradle" counters
"""

import os, re, csv, random, stat, shutil, subprocess, requests
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
from dotenv import load_dotenv, set_key

# ========= CONFIG =========
MAX_PROJECTS = 4697
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# ---- Inputs / Outputs ----
csv_path = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8_Test\URL_List.csv")
base_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8_Test")

clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
config_bucket = base_dir / "All_Config_Files"
tests_bucket = base_dir / "All_Test_Files"
commits_dir = base_dir / "Commits"
git_metadata_dir = base_dir / "Git_Metadata"

# Global CSV index
flat_index_csv = config_bucket.parent / "All_Config_Index.csv"

metadata_path = base_dir / "Project_Metadata.csv"
list_of_config_path = base_dir / "List_of_Config.csv"

# ========= ENV / TOKENS =========
load_dotenv(ENV_FILE)
TOKENS = [os.getenv(f'GITHUB_TOKEN_{i}') for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]
if not TOKENS:
    print("⚠️ Warning: No GitHub tokens found in All_tokens.env (API metadata might be rate limited).")
token_index = 0

START_NUMBER = int(os.getenv("START_NUMBER_2") or "1")
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST_2", "").strip()

# ========= Ensure folders =========
for path in [clone_dir, commits_dir, cloned_sample_dir, git_metadata_dir, config_bucket, tests_bucket]:
    path.mkdir(parents=True, exist_ok=True)

# ========= CI platform patterns (STRICT YAML LOCATIONS) =========
ci_patterns = {
    r'\.travis\.ya?ml$': 'Travis_CI',
    r'\.appveyor\.ya?ml$': 'AppVeyor',
    r'appveyor\.ya?ml$': 'AppVeyor',
    r'circle\.ya?ml$': 'Circle_CI',
    r'\.circleci/config\.(yml|yaml)$': 'Circle_CI',
    r'azure-pipelines\.ya?ml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',
    r'bitbucket-pipelines\.ya?ml$': 'Bitbucket',
    r'\.gitlab-ci\.ya?ml$': 'GitLab',
    r'Jenkinsfile\.ya?ml$': 'Jenkins',
    r'bitrise\.ya?ml$': 'Bitrise',
    r'bamboo\.ya?ml$': 'Bamboo',
    r'codeship-services\.ya?ml$': 'Codeship',
    r'\.gocd\.ya?ml$': 'GoCD',
    r'\.cirrus\.ya?ml$': 'Cirrus',
    r'wercker\.ya?ml$': 'Wercker',
    r'semaphore\.ya?ml$': 'Semaphore',
    r'codemagic\.ya?ml$': 'Nevercode',
}

# ========= Inclusion patterns (expanded) =========
# Reasonable coverage for modern projects:
# - Any build script named build.gradle(.kts)
# - Any settings.gradle(.kts)
# - gradle.properties
# - Version catalogs (libs.versions.toml + other *.toml under gradle/)
# - Convention plugin scripts (ANY *.gradle/.gradle.kts under buildSrc/** or gradle/plugins/**)
# - Convention plugin code (.kt/.java) under buildSrc/** or gradle/plugins/**
# - Any AndroidManifest.xml (any source set)
# - ONLY androidTest tree for tests
LIKELY_CI_JSON = {"android-studio-loading.json", "saucectl.config.json", "firebase.json", "test-lab.json"}

INCLUDE_PATTERNS = (
    # Module/root build files named build.gradle(.kts)
    re.compile(r'(?:^|.*/)build\.gradle(\.kts)?$', re.I),

    # settings.gradle(.kts)
    re.compile(r'(?:^|.*/)settings\.gradle(\.kts)?$', re.I),

    # gradle.properties
    re.compile(r'(?:^|.*/)gradle\.properties$', re.I),

    # Version catalogs (default + any *.toml under gradle/)
    re.compile(r'(?:^|.*/)gradle/libs\.versions\.toml$', re.I),
    re.compile(r'(?:^|.*/gradle/)[^/]+\.toml$', re.I),

    # Convention plugin scripts anywhere under buildSrc/** or gradle/plugins/**
    re.compile(r'(?:^|.*/)(buildSrc|gradle/plugins)/.*\.(gradle|gradle\.kts)$', re.I),

    # Convention plugin code under buildSrc/** or gradle/plugins/** (.kt/.java)
    re.compile(r'(?:^|.*/)(buildSrc|gradle/plugins)/.*\.(kt|java)$', re.I),

    # Android manifests (any source set)
    re.compile(r'(?:^|.*/)AndroidManifest\.xml$', re.I),

    # ONLY androidTest tree (instrumentation tests)
    re.compile(r'(?:^|.*/)src/androidTest/.*', re.I),

    # Shell scripts anywhere
    re.compile(r'(?:^|.*/)[^/]+\.sh$', re.I),

    # JSON (filtered later unless under workflows or known names)
    re.compile(r'(?:^|.*/)[^/]+\.json$', re.I),

    # Jenkinsfile (declarative pipelines)
    re.compile(r'(?:^|.*/)Jenkinsfile$', re.I),
)

# ========= Trigger & Env keyword detectors =========
TRIGGER_REGEX = re.compile(
    r'(?:^|\s)(?:\.\/)?gradle(?:w)?\s+.*connected(?:Android|Debug|Release)?AndroidTest'
    r'|connectedCheck\b|adb\s+shell\s+am\s+instrument\b|gcloud\s+firebase\s+test\s+android\s+run\b',
    re.I
)
EXEC_ENV_REGEX = re.compile(
    r'reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|'
    r'\bsdkmanager\b|\bavdmanager\b|\bemulator\b|'
    r'browserstack/|appcenter\s+test\s+run|saucectl|'
    r'gradle\s+managed\s+devices|managedDevices|managedVirtualDevice|cleanManagedDevices',
    re.I
)

# ========= CSV setup =========
if not flat_index_csv.exists():
    with open(flat_index_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "owner","repo","repo_url","default_branch","commit_sha",
            "relative_path","filename","flat_filename","ci_platform","html_url",
            "saved_to","bucket","components"
        ])
        writer.writeheader()

# ========= Helpers =========
def run(cmd, cwd=None, check=True):
    return subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=check)

def strict_yaml_match(rel_path: str):
    """Return (is_strict: bool, platform: str) for YAML files only."""
    p = rel_path.replace("\\", "/")
    for pattern, platform in ci_patterns.items():
        if re.search(pattern, p, re.IGNORECASE):
            return True, platform
    return False, "CI_YAML"

def is_androidtest_file(rel_path: str) -> bool:
    """Instrumentation test sources ONLY: under src/androidTest/**."""
    return "/src/androidtest/" in rel_path.replace("\\", "/").lower()

def parse_owner_repo(url: str):
    parts = urlparse(url)
    if parts.netloc.lower() != "github.com":
        raise ValueError("Only github.com URLs supported")
    pieces = parts.path.strip("/").split("/")
    if len(pieces) < 2:
        raise ValueError("Invalid GitHub URL")
    return pieces[0], pieces[1].replace(".git", "")

def detect_default_branch_via_git(repo_url: str) -> str:
    p = run(["git", "ls-remote", "--symref", repo_url, "HEAD"])
    for line in p.stdout.splitlines():
        s = line.strip()
        if s.startswith("ref: ") and s.endswith("HEAD"):
            ref = s.split()[1]
            if ref.startswith("refs/heads/"):
                return ref.split("/", 2)[2]
    for guess in ("main", "master"):
        try:
            run(["git", "ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            pass
    raise RuntimeError("Could not determine default branch (ls-remote)")

def shallow_clone_branch(repo_url: str, dest: Path, branch: str):
    dest.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch, repo_url, str(dest)])

def matches_path_non_yaml(rel_path: str, filename: str) -> bool:
    """Apply INCLUDE_PATTERNS for non-YAML, with JSON filtering."""
    rp = rel_path.replace("\\", "/")
    rp_lower = rp.lower()
    for pat in INCLUDE_PATTERNS:
        if pat.search(rp):
            if filename.lower().endswith(".json"):
                name_only = Path(rel_path).name.lower()
                if (name_only not in LIKELY_CI_JSON) and (not rp_lower.startswith(".github/workflows/")):
                    return False
            return True
    return False

def detect_ci_platform(rel_path: str, filename: str) -> str:
    """Heuristic platform label (non-YAML)."""
    p = rel_path.replace("\\", "/")
    f = filename.lower()
    pl = p.lower()

    # Keep strict alignment if a non-yaml lands in a strict path (rare but ok)
    for pattern, platform in ci_patterns.items():
        if re.search(pattern, p, re.IGNORECASE):
            return platform

    if pl.startswith(".github/workflows/"):               return "GitHub_Actions"
    if "/.circleci/" in pl or "circle.yml" in pl:         return "Circle_CI"
    if "azure" in pl or f == "azure-pipelines.yml":       return "Azure_Pipelines"
    if "/.gitlab/" in pl or f == ".gitlab-ci.yml":        return "GitLab"
    if "bitrise" in pl or "bitrise" in f:                 return "Bitrise"
    if "jenkins" in pl or f in {"jenkinsfile", "jenkinsfile.yml"}:
                                                           return "Jenkins"
    if "bitbucket-pipelines" in pl:                       return "Bitbucket"
    if "semaphore" in pl:                                 return "Semaphore"
    if "wercker" in pl:                                   return "Wercker"
    if "codeship" in pl:                                  return "Codeship"
    if "cirrus" in pl:                                    return "Cirrus"
    if ".gocd" in pl or "gocd" in pl:                     return "GoCD"
    if "bamboo" in pl:                                    return "Bamboo"
    if "codemagic" in pl:                                 return "Nevercode"
    if "appveyor" in pl or f in {"appveyor.yml", ".appveyor.yml"}:
                                                           return "AppVeyor"
    if "travis" in pl or f == ".travis.yml":              return "Travis_CI"

    if f.endswith(".sh"):                                 return "Shell"
    if f.endswith(".gradle") or f.endswith(".kts"):       return "Gradle"
    if f.endswith(".toml"):                               return "Gradle"
    if f == "androidmanifest.xml":                        return "Manifest"
    if "src/androidtest/" in pl:                          return "AndroidTest"
    return "Other"

def classify_components(rel_path: str, filename: str, content: str) -> str:
    rp = rel_path.replace("\\", "/").lower()
    fn = filename.lower()
    comps = set()

    # (1) Test Definition (wiring & tests)
    if fn.endswith((".gradle", ".gradle.kts", ".toml")) or \
       fn in {"androidmanifest.xml"} or \
       fn in {"gradle.properties", "settings.gradle", "settings.gradle.kts"} or \
       "/src/androidtest/" in rp or \
       re.search(r'(?:^|/)(buildsrc|gradle/plugins)/', rp):
        comps.add("1")

    # (2) Device Setup (CI env) — YAML/SH or emulator/setup hints
    if EXEC_ENV_REGEX.search(content or "") or \
       fn.endswith((".yml", ".yaml", ".sh")) or \
       "/.github/workflows/" in rp or \
       "manageddevices" in (content or "").lower():
        comps.add("2")

    # (3) Test Trigger
    if TRIGGER_REGEX.search(content or ""):
        comps.add("3")

    # Fallbacks
    if not comps:
        if fn.endswith((".yml", ".yaml", ".sh")):
            comps.add("2")
        elif "/src/androidtest/" in rp:
            comps.add("1")

    return ";".join(sorted(comps)) if comps else ""

def get_count(api_url, headers):
    per_page = 100
    page = 1
    total_items = 0
    try:
        while True:
            response = requests.get(api_url, headers=headers, params={"per_page": per_page, "page": page}, timeout=30)
            if response.status_code != 200:
                break
            items = response.json()
            if not isinstance(items, list):
                break
            total_items += len(items)
            if len(items) < per_page:
                break
            page += 1
    except Exception:
        pass
    return total_items

def extract_commit_metadata(repo_path: Path, output_folder: Path):
    try:
        cmd_hashes = ["git", "-C", str(repo_path), "log", "--pretty=format:%H"]
        result_hashes = subprocess.run(cmd_hashes, capture_output=True, text=True, check=True)
        commit_hashes = result_hashes.stdout.strip().split("\n")

        rows = []
        for commit in commit_hashes:
            cmd_metadata = ["git", "-C", str(repo_path), "show", "--quiet",
                            f"--pretty=format:%H|%an|%ae|%ad|%s", "--date=iso", commit]
            result_metadata = subprocess.run(cmd_metadata, capture_output=True, text=True)
            if not result_metadata.stdout:
                continue
            parts = result_metadata.stdout.strip().split("|", maxsplit=4)
            if len(parts) < 5:
                continue

            cmd_files = ["git", "-C", str(repo_path), "show", "--name-only", "--pretty=format:", commit]
            result_files = subprocess.run(cmd_files, capture_output=True, text=True, check=True)
            changed_files = [f.strip() for f in result_files.stdout.strip().split("\n") if f.strip()]

            lower_changed = [c.lower() for c in changed_files]
            count_androidTest = sum("src/androidtest/" in c for c in lower_changed)
            count_github_workflows = sum(".github/workflows" in c for c in lower_changed)
            count_gradle = sum(c.endswith(".gradle") or c.endswith(".gradle.kts") or c.endswith(".toml") for c in lower_changed)

            rows.append({
                "commit_hash": parts[0],
                "author_name": parts[1],
                "author_email": parts[2],
                "commit_date": parts[3],
                "commit_message": parts[4],
                "touches_androidTest": count_androidTest > 0,
                "count_androidTest": count_androidTest,
                "touches_github_workflows": count_github_workflows > 0,
                "count_github_workflows": count_github_workflows,
                "touches_gradle": count_gradle > 0,
                "count_gradle": count_gradle
            })

        if rows:
            dfc = pd.DataFrame(rows)
            output_folder.mkdir(parents=True, exist_ok=True)
            flat_filename = f"{repo_path.name}__GitMetadata++contributors_commits.csv"
            dfc.to_csv(output_folder / flat_filename, index=False)
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to extract commit data for {repo_path.name}: {e}")

# ========= Load URL list =========
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# ========= Sampling =========
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# ========= Process =========
review_status_rows = []
CLONE_FAILURE_COLUMNS = ["repo_index", "repo_name", "github_url", "error_message"]

for i in range(START_NUMBER - 1, min(len(df), MAX_PROJECTS)):
    url = df.iloc[i]['github_url']
    owner_repo = urlparse(url).path.strip("/").split("/")
    if len(owner_repo) < 2:
        continue
    owner, project = owner_repo[0], owner_repo[1].replace(".git", "")
    repo_index = str(i).zfill(4)
    repo_name_tag = f"{repo_index}.{owner}.{project}"
    repo_path = clone_dir / repo_name_tag
    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name_tag}...")

    # Clone default branch only
    try:
        default_branch = detect_default_branch_via_git(url)
        print(f"📌 Default branch: {default_branch}")
        shallow_clone_branch(url, repo_path, default_branch)
        print("✅ Clone complete")
    except Exception as e:
        error_message = (str(e) or "Unknown error").strip()
        print(f"❌ Clone failed for {repo_name_tag}\n{error_message}")
        review_status_rows.append({"html_url": url.strip(), "clone_status": "no", "yml_detected": "no"})
        pd.DataFrame([review_status_rows[-1]]).to_csv(
            base_dir / "Clone_Status.csv", mode='a', header=not (base_dir / "Clone_Status.csv").exists(), index=False
        )
        fail_row = {"repo_index": repo_index, "repo_name": repo_name_tag, "github_url": url.strip(), "error_message": error_message}
        fail_path = base_dir / "Clone_Failures.csv"
        pd.DataFrame([fail_row])[CLONE_FAILURE_COLUMNS].to_csv(
            fail_path, mode='a', header=not fail_path.exists(), index=False
        )
        continue

    # Commit count + SHA + commit metadata with touches counters
    try:
        local_commit_count = int(subprocess.run(['git', '-C', str(repo_path), 'rev-list', '--count', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    except subprocess.CalledProcessError:
        local_commit_count = 0
        print(f"⚠️ Could not get commit count for {repo_name_tag}")

    try:
        head_sha = subprocess.run(['git', '-C', str(repo_path), 'rev-parse', '--verify', 'HEAD'],
                                  capture_output=True, text=True, check=True).stdout.strip()
    except subprocess.CalledProcessError:
        head_sha = ""

    if local_commit_count > 0:
        extract_commit_metadata(repo_path, git_metadata_dir)

    # Walk files and extract
    any_yml = False
    legacy_config_rows = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            # YAML: keep ONLY strict CI-located YAMLs
            if file_lower.endswith(('.yml', '.yaml')):
                is_strict, platform = strict_yaml_match(rel_path)
                if not is_strict:
                    continue  # skip heuristic YAML entirely
                ci_platform = platform
                is_test_source = is_androidtest_file(rel_path)
                bucket = "All_Test_Files" if is_test_source else "All_Config_Files"
                flat_filename = f"{owner}.{project}__{ci_platform}++{file_lower}"
                dest_path = (tests_bucket if is_test_source else config_bucket) / flat_filename
                dest_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(file_path, dest_path)
                any_yml = True

                try:
                    content = file_path.read_text(encoding="utf-8", errors="ignore")
                except Exception:
                    content = ""
                components = classify_components(rel_path, file_lower, content)

            else:
                # Non-YAML: use the expanded inclusion + heuristics
                if not matches_path_non_yaml(rel_path, file):
                    continue
                try:
                    content = file_path.read_text(encoding="utf-8", errors="ignore")
                except Exception:
                    content = ""

                ci_platform = detect_ci_platform(rel_path, file)
                is_test_source = is_androidtest_file(rel_path)
                bucket = "All_Test_Files" if is_test_source else "All_Config_Files"
                flat_filename = f"{owner}.{project}__{ci_platform}++{file_lower}"
                dest_path = (tests_bucket if is_test_source else config_bucket) / flat_filename
                dest_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(file_path, dest_path)
                components = classify_components(rel_path, file_lower, content)

            # Write unified CSV index (no ci_source)
            with open(flat_index_csv, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=[
                    "owner","repo","repo_url","default_branch","commit_sha",
                    "relative_path","filename","flat_filename","ci_platform","html_url",
                    "saved_to","bucket","components"
                ])
                writer.writerow({
                    "owner": owner,
                    "repo": project,
                    "repo_url": url.strip(),
                    "default_branch": default_branch,
                    "commit_sha": head_sha,
                    "relative_path": rel_path,
                    "filename": file,
                    "flat_filename": flat_filename,
                    "ci_platform": ci_platform,
                    "html_url": f"https://github.com/{owner}/{project}/blob/{default_branch}/{rel_path}",
                    "saved_to": str(dest_path),
                    "bucket": bucket,
                    "components": components
                })

            # Maintain legacy List_of_Config.csv
            legacy_config_rows.append({
                "html_url": url.strip().rstrip('/'),
                "repo_name": repo_name_tag,
                "config_file_path": flat_filename,
                "original_rel_path": rel_path,
                "file_name": file,
                "file_type": file_lower.split('.')[-1]
            })

    # Clone status
    review_status_row = {"html_url": url.strip(), "clone_status": "yes", "yml_detected": "yes" if any_yml else "no"}
    pd.DataFrame([review_status_row]).to_csv(
        base_dir / "Clone_Status.csv", mode='a', header=not (base_dir / "Clone_Status.csv").exists(), index=False
    )

    # Persist legacy List_of_Config.csv
    if legacy_config_rows:
        ldf = pd.DataFrame(legacy_config_rows)
        if list_of_config_path.exists():
            ldf.to_csv(list_of_config_path, mode='a', header=False, index=False)
        else:
            ldf.to_csv(list_of_config_path, mode='w', header=True, index=False)

    # Project metadata + paginated counts + contributors
    try:
        headers = {}
        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        base_api = f"https://api.github.com/repos/{owner}/{project}"
        r = requests.get(base_api, headers=headers, timeout=30)
        data = r.json() if r.status_code == 200 else {}

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        contributors_count = get_count(f"{base_api}/contributors", headers)

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        pulls_count = get_count(f"{base_api}/pulls?state=all", headers)

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        commits_count = get_count(f"{base_api}/commits", headers)

        metadata_row = {
            "html_url": url,
            "repo_index": repo_index,
            "repo_name": repo_name_tag,
            "id": data.get("id"),
            "name": data.get("name"),
            "full_name": data.get("full_name"),
            "owner": data.get("owner", {}).get("login") if data.get("owner") else None,
            "private": data.get("private"),
            "fork": data.get("fork"),
            "created_at": data.get("created_at"),
            "updated_at": data.get("updated_at"),
            "pushed_at": data.get("pushed_at"),
            "homepage": data.get("homepage"),
            "size": data.get("size"),
            "stargazers_count": data.get("stargazers_count"),
            "language": data.get("language"),
            "forks_count": data.get("forks_count"),
            "open_issues_count": data.get("open_issues_count"),
            "license": data.get("license", {}).get("name") if data.get("license") else None,
            "topics": ", ".join(data.get("topics", [])) if data.get("topics") else None,
            "visibility": data.get("visibility"),
            "default_branch": data.get("default_branch"),
            "has_issues": data.get("has_issues"),
            "has_projects": data.get("has_projects"),
            "has_downloads": data.get("has_downloads"),
            "has_wiki": data.get("has_wiki"),
            "has_pages": data.get("has_pages"),
            "archived": data.get("archived"),
            "disabled": data.get("disabled"),
            "allow_forking": data.get("allow_forking"),
            "is_template": data.get("is_template"),
            "web_commit_signoff_required": data.get("web_commit_signoff_required"),
            "contributors": contributors_count,
            "pull_requests": pulls_count,
            "commits_GitAPI": commits_count,
            "local_commit_count": local_commit_count
        }
        mdf = pd.DataFrame([metadata_row])
        if metadata_path.exists():
            mdf.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            mdf.to_csv(metadata_path, mode='w', header=True, index=False)

        if TOKENS:
            headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
            token_index += 1
        contrib_url = f"{base_api}/contributors"
        r_contrib = requests.get(contrib_url, headers=headers, timeout=30)
        if r_contrib.status_code == 200:
            contributor_logins = [c['login'] for c in r_contrib.json()]
            contributors_text = "\n".join(contributor_logins)
            contributors_filename = f"{owner}.{project}__Contributors++list.txt"
            contributors_path = commits_dir / contributors_filename
            with open(contributors_path, "w", encoding="utf-8") as f:
                f.write(contributors_text)

    except Exception as e:
        print(f"⚠️ Metadata or contributors error for {repo_name_tag}: {e}")

    # Cleanup or keep sample
    # try:
    #     if i in sample_indices_to_keep:
    #         dest_path = cloned_sample_dir / repo_path.name
    #         if dest_path.exists():
    #             shutil.rmtree(dest_path, ignore_errors=True)
    #         shutil.move(str(repo_path), str(dest_path))
    #         print(f"📆 Sample repo moved to: {dest_path}")
    #     else:
    #         shutil.rmtree(repo_path, onerror=lambda f,p,e: (os.chmod(p, stat.S_IWRITE), f(p)))
    #         print(f"🕵️ Deleted cloned repo: {repo_name_tag}")
    # except Exception as e:
    #     print(f"❌ Error handling repo folder for {repo_name_tag}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

# Final dedupes
for p in [base_dir / "List_of_Config.csv",
          base_dir / "Clone_Failures.csv",
          base_dir / "Project_Metadata.csv",
          base_dir / "Clone_Status.csv"]:
    if p.exists():
        try:
            dfp = pd.read_csv(p)
            dfp.drop_duplicates().to_csv(p, index=False)
        except Exception:
            pass

print("\n✅ Process complete.")


🔁 Loaded SAMPLE_LIST from .env with 1 indices.

🔍 [1/1] Processing 0000.BennyKok.PxerStudio...
📌 Default branch: master
✅ Clone complete

✅ Process complete.
